# Retail Demand Forecasting: Data to Evaluation

This notebook builds and evaluates a daily product-store demand forecast from the M5 dataset. Run cells from top to bottom from the repository root. The six sections make the workflow explicit: data cleaning, exploratory data analysis (EDA), preprocessing, model training, evaluation, and interpretation. The final cells save the model and demonstrate a forecast.

The experiment uses a chronological 70/30 split. The final 30% of dates stay out of training and model selection. A seeded sample (`MAX_SERIES = 1000`) is available for a faster end-to-end run; use `0` for all series.


## Workflow at a glance

| Section | What happens |
| --- | --- |
| 1. Data cleaning | Check required files, schema coverage, missing values, invalid sales, duplicates, and price gaps. Preserve valid zero-sales days and flag unavailable prices. |
| 2. EDA | Inspect sales volume, zero-sales frequency, time patterns, and price coverage. |
| 3. Preprocessing | Encode categories, align calendar and prices, build lag features, and split by date. |
| 4. Model training | Select boosting rounds on training-only validation dates, then fit on the full training partition. |
| 5. Evaluation | Recursively forecast held-out dates and compare against simple baselines. |
| 6. Interpretation | Summarize errors, uncertainty, performance by demand volatility, and feature importance. |

The target is daily unit sales for each product-store series. Error scores measure forecasting accuracy; they do not directly measure inventory cost or business benefit.


In [ ]:
from pathlib import Path
import sys

# Run from the repository root. In Colab, set this to the extracted project directory.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'src' / 'forecast_holdout.py').exists():
    raise FileNotFoundError(f'Project root not found at {PROJECT_DIR}. Set PROJECT_DIR to the repository folder.')
sys.path.insert(0, str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / 'data' / 'raw'
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'holdout_70_30'
MAX_SERIES = 0  # 0 = all series; set 1000 for a quick end-to-end run
BOOTSTRAP_REPLICATES = 1000
SEED = 42
print('Project:', PROJECT_DIR)
print('Data:', DATA_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
%pip install -r requirements.txt

In [ ]:
import hashlib
import json
import platform
from datetime import datetime, timezone

import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.forecast_experiments import CODES, HORIZON, SEED as MODEL_SEED, TRAIN_DAYS, fit, load_data, make_features, volatility
from src.forecast_holdout import (TRAIN_FRACTION, STATIC, MODELS, compact_feature_dtypes, model_parameters, rolling_holdout_forecasts, summarize_metrics, bootstrap_comparisons)
assert SEED == MODEL_SEED, 'Keep the reproducibility seed aligned with the training helpers.'
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'lightgbm': lgb.__version__})

## 1. Data cleaning

Load the source tables through the shared project loader. It checks that sales and calendar files exist, confirms the calendar covers every sales day, sorts daily columns, and aligns weekly prices to each product-store series. The checks below make remaining data quality issues visible before modeling.

M5 zero sales are valid observations and stay zero. Missing scheduled prices are retained as missing values and represented by a `price_missing` feature; LightGBM can use the missing numeric price values. Negative or missing sales and duplicate series identifiers are treated as data errors and stop the workflow.


In [ ]:
required_files = ['sales_train_validation.csv', 'calendar.csv', 'sell_prices.csv']
missing = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing {missing} under {DATA_DIR}. See README.md for data setup.')
meta, y, codes, days, calendar, sales_file, prices, price_file = load_data(DATA_DIR, MAX_SERIES)
if prices is None:
    raise ValueError('sell_prices.csv is required for this price-aware model.')

quality = {
    'series_rows': len(meta),
    'unique_series_ids': meta['id'].nunique(),
    'daily_columns': len(days),
    'missing_sales_values': int(np.isnan(y).sum()),
    'negative_sales_values': int((y < 0).sum()),
    'missing_calendar_features': int(calendar.isna().sum().sum()),
    'missing_scheduled_prices': int(np.isnan(prices).sum()),
    'zero_sales_share': float((y == 0).mean()),
}
display(pd.Series(quality, name='value').to_frame())
if quality['series_rows'] != quality['unique_series_ids']:
    raise ValueError('Duplicate product-store series identifiers were found.')
if quality['missing_sales_values'] or quality['negative_sales_values']:
    raise ValueError('Sales contain missing or negative values; inspect the source before training.')
if quality['missing_calendar_features']:
    raise ValueError('Calendar features contain missing values after alignment.')
print(f'Loaded {len(meta):,} series ? {len(days):,} daily observations.')


## 2. Exploratory data analysis (EDA)

These summaries show how demand is distributed and how it changes over time. The overall series totals can be dominated by large products, so the log scale on the histogram helps reveal the long tail. All plots use the loaded data and fixed labels, so reruns are comparable.


In [ ]:
# Aggregate daily sales and product-store demand totals for descriptive inspection.
daily_demand = y.sum(axis=0, dtype=np.float64)
series_totals = y.sum(axis=1, dtype=np.float64)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(pd.to_datetime(calendar['date']), daily_demand, linewidth=0.8)
axes[0].set(title='Total daily sales across selected series', xlabel='Date', ylabel='Units sold')
axes[0].grid(alpha=0.25)
axes[1].hist(series_totals, bins=50, log=True, color='#4472C4', edgecolor='white')
axes[1].set(title='Total sales per product-store series', xlabel='Units sold over available history', ylabel='Series (log scale)')
axes[1].grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

print('Daily demand:')
display(pd.Series(daily_demand).describe(percentiles=[.1, .5, .9]).to_frame('units'))
print('Missing scheduled price share:', f'{np.isnan(prices).mean():.1%}')
display(meta[['id', 'item_id', 'store_id', 'cat_id', 'dept_id', 'state_id']].head())


## 3. Preprocessing

The loader aligns calendar fields and weekly prices to daily observations and encodes store, category, department, and state as categorical codes. Features are created from recent sales (lags and rolling statistics), date/calendar signals, category codes, and scheduled prices. The first 28 days are reserved as feature warm-up because lag features need past observations.

We split by date: the earliest 70% is used for training and the latest 30% is reserved for final evaluation. A chronological split prevents future sales from leaking into the training examples.


### Build the time split

The model uses eligible training labels after the 28-day feature warm-up. The test partition remains untouched until the evaluation section.


In [ ]:
split_day = int(np.floor(y.shape[1] * TRAIN_FRACTION))
test_horizon = y.shape[1] - split_day
if split_day < max(TRAIN_DAYS, 52 * 7) + HORIZON or test_horizon < HORIZON:
    raise ValueError('Dataset is too short for the configured training and holdout windows.')
history = y[:, :split_day]
cutoff = split_day - 1
train_days = np.arange(28, split_day, dtype=np.int32)  # first 28 days are feature warm-up
print(f'Train: {days[0]} through {days[cutoff]} ({split_day:,} days, {TRAIN_FRACTION:.0%})')
print(f'Test:  {days[split_day]} through {days[-1]} ({test_horizon:,} days, {1-TRAIN_FRACTION:.0%})')
print(f'Eligible training labels: {len(meta) * len(train_days):,}')


## 4. Model training

LightGBM learns an ensemble of decision trees. A 28-day validation slice inside the training partition selects the number of boosting rounds. We then refit using all eligible dates in the 70% training partition. The held-out test period is not used for either choice.


In [ ]:
selector = fit(history, cutoff, calendar, codes, prices)
selected_rounds = max(1, int(selector.best_iteration or selector.current_iteration()))
del selector
train_features = compact_feature_dtypes(make_features(history, train_days, calendar, codes, prices))
train_labels = history[:, train_days].T.reshape(-1)
train_data = lgb.Dataset(train_features, label=train_labels, categorical_feature=CODES, free_raw_data=True)
model = lgb.train(model_parameters(), train_data, num_boost_round=selected_rounds)
del train_features, train_labels, train_data
print(f'Final model fitted with {selected_rounds} boosting rounds.')

## 5. Evaluation

The held-out period is forecast in rolling windows of up to 28 days. Within each window, predictions are fed back as history for later forecast days. After each window ends, its actual sales become available as history for the next origin. The current window's actual sales never enter that window's predictions.


In [ ]:
predictions = rolling_holdout_forecasts(model, y, split_day, test_horizon, calendar, codes, prices)
actual = y[:, split_day:]
test_days = days[split_day:]
volatility_groups, zero_week_share = volatility(history)
print('Prediction arrays:', {name: values.shape for name, values in predictions.items()})

We compare the LightGBM model with two simple baselines: repeating the same weekday's sales from the previous week, and averaging that weekday over the previous four weeks. WAPE measures total absolute error relative to total demand; MAE is the average absolute error per forecast; RMSE penalizes large errors more heavily. Lower is better. Bootstrap intervals quantify uncertainty in the WAPE improvement across product-store series.


In [ ]:
metrics = summarize_metrics(actual, predictions, volatility_groups, test_days)
series_metrics = pd.DataFrame({
    'series_id': meta['id'].to_numpy(),
    'volatility_group': volatility_groups,
    'zero_week_share': zero_week_share,
    'actual_demand_sum': actual.sum(axis=1),
    **{f'absolute_error_{name}': np.abs(actual - forecast).sum(axis=1) for name, forecast in predictions.items()},
})
bootstrap = bootstrap_comparisons(series_metrics, BOOTSTRAP_REPLICATES)
display(metrics[metrics.volatility_group == 'all'][['model', 'WAPE', 'MAE', 'RMSE']].sort_values('WAPE'))
display(metrics[metrics.volatility_group != 'all'][['model', 'volatility_group', 'WAPE', 'MAE', 'RMSE']])
display(bootstrap)

## 6. Interpretation

Read the overall metrics first, then compare results across volatility groups. A positive WAPE improvement means the candidate model has lower error than the reference. If its 95% bootstrap interval crosses zero, the data do not clearly establish an improvement across series. Feature importance reports which inputs the fitted trees used most; it describes model behavior, not causal effects.


In [ ]:
# Compare the model to the stronger of the two baselines on overall WAPE.
overall = metrics.loc[metrics.volatility_group == 'all'].set_index('model')
model_name = 'lightgbm_recursive_price'
baseline_names = ['seasonal_naive_7d', 'four_week_weekday_mean']
best_baseline = overall.loc[baseline_names, 'WAPE'].idxmin()
model_wape = float(overall.loc[model_name, 'WAPE'])
baseline_wape = float(overall.loc[best_baseline, 'WAPE'])
improvement_pct = (baseline_wape - model_wape) / baseline_wape * 100 if baseline_wape else float('nan')
print(f"Overall WAPE: {model_name}={model_wape:.4f}; best baseline ({best_baseline})={baseline_wape:.4f}.")
print(f'Relative WAPE improvement over best baseline: {improvement_pct:.2f}% (positive favors LightGBM).')

comparison = bootstrap[
    (bootstrap.volatility_group == 'all') &
    (bootstrap.reference_model == best_baseline) &
    (bootstrap.candidate_model == model_name)
]
if not comparison.empty:
    row = comparison.iloc[0]
    print(f"Bootstrap 95% interval for WAPE improvement: [{row['improvement_95CI_low']:.4f}, {row['improvement_95CI_high']:.4f}].")
    print(f"Bootstrap probability improvement is positive: {row['bootstrap_probability_improvement_positive']:.1%}.")
else:
    print('No matching bootstrap comparison was produced; check the selected sample and metric outputs.')

importance = pd.DataFrame({
    'feature': model.feature_name(),
    'gain': model.feature_importance(importance_type='gain'),
}).sort_values('gain', ascending=False).head(15)
importance['gain_share'] = importance['gain'] / max(float(importance['gain'].sum()), 1.0)
display(importance)
importance.sort_values('gain').plot.barh(x='feature', y='gain_share', figsize=(8, 5), legend=False, color='#4472C4')
plt.title('Top 15 features by share of gain importance')
plt.xlabel('Share of gain importance')
plt.ylabel('')
plt.tight_layout()
plt.show()


## Save the model and evaluation outputs

Save the trained model, metadata, score tables, and a manifest with the date split, input checksums, random seed, and library versions. This makes the run auditable and supports later inference.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
series_metrics.to_csv(OUTPUT_DIR / 'series_metrics.csv', index=False)
bootstrap.to_csv(OUTPUT_DIR / 'bootstrap_comparison.csv', index=False)
model_path = OUTPUT_DIR / 'lightgbm_price.txt'
model.save_model(str(model_path))
category_maps = {code: [str(value) for value in pd.Categorical(meta[source]).categories] for code, source in zip(CODES, STATIC)}
metadata = {
    'model': 'lightgbm_recursive_price', 'model_file': model_path.name,
    'feature_names': model.feature_name(), 'categorical_features': CODES,
    'category_code_maps': category_maps, 'horizon_days': HORIZON, 'history_days_required': 28,
    'selected_boosting_rounds': selected_rounds, 'training_fraction': TRAIN_FRACTION,
    'training_days': split_day, 'test_days': test_horizon, 'series_evaluated': len(meta),
    'seed': MODEL_SEED, 'sales_file_sha256': hashlib.sha256(sales_file.read_bytes()).hexdigest(),
    'sell_prices_file_sha256': hashlib.sha256(price_file.read_bytes()).hexdigest(),
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'lightgbm': lgb.__version__,
}
manifest = {
    'evaluation': 'chronological 70/30 holdout', 'train_fraction': TRAIN_FRACTION,
    'test_fraction': round(1 - TRAIN_FRACTION, 2), 'forecast_window_days': HORIZON,
    'rolling_forecast_origins': int(np.ceil(test_horizon / HORIZON)),
    'train_start_day': days[0], 'train_end_day': days[cutoff],
    'test_start_day': days[split_day], 'test_end_day': days[-1],
    'training_days': split_day, 'test_days': test_horizon,
    'training_observations': int(len(meta) * len(train_days)), 'series_evaluated': len(meta),
    'selected_boosting_rounds': selected_rounds, 'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'seed': MODEL_SEED, 'sales_file_sha256': metadata['sales_file_sha256'],
    'sell_prices_file_sha256': metadata['sell_prices_file_sha256'],
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'lightgbm': lgb.__version__, 'run_timestamp_utc': datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved model:', model_path)
print('Saved metrics:', OUTPUT_DIR / 'metrics.csv')

## Forecast one example series with the saved model

The inference helper receives the latest sales history and future calendar and price information. The true held-out sales are displayed only for inspection after the forecast is generated.


In [ ]:
from src.forecast_inference import forecast_series, load_model_bundle
saved_model, saved_metadata = load_model_bundle(OUTPUT_DIR)
example_series = 0
example_forecast = forecast_series(
    saved_model, saved_metadata,
    sales_history=y[example_series, split_day - 28:split_day],
    static_metadata=meta.iloc[example_series][STATIC].to_dict(),
    future_calendar=calendar.iloc[split_day:split_day + 28].to_dict(orient='records'),
    recent_prices=prices[example_series, split_day - 28:split_day],
    future_prices=prices[example_series, split_day:split_day + 28],
)
example_result = pd.DataFrame({
    'day_id': days[split_day:split_day + 28],
    'actual_units': actual[example_series, :28],
    'predicted_units': example_forecast,
})
print('Example product/store:', meta.iloc[example_series]['id'])
display(example_result.head(10))

## Saved outputs

The output folder contains the trained model, metadata describing its features and category codes, evaluation tables, and a run manifest. The scores describe this historical split and dataset; they are not a guarantee of future accuracy or another retailer's results.


## Reproduce this notebook

From the repository root, open this notebook in Jupyter and run all cells in order. Use the project environment and packages listed in `requirements.txt`. Keep `SEED = 42` for reproducible sampling and training behavior. Set `MAX_SERIES = 1000` for a smaller pipeline run or `0` for all series. Full-population training can require substantial memory and time. The manifest records input file hashes and package versions for each run.
